# Day 15 — RAG Part 2: Generation
## 30 Days of AI: From NLP to LLMs

---

On Day 14 you built the retrieval half of RAG: documents were
loaded, chunked, embedded, and stored in a vector store with
full metadata support. You can now query that store and get
back the most relevant passages for any question.

Today you connect retrieval to generation. You will write the
RAG prompt template, pipe retrieved context into an LLM, handle
source citations, and evaluate the full pipeline end-to-end.
By the end of today you have a complete, working RAG system —
the same architecture used in production document Q&A products.

---

### What You Will Learn Today

- The RAG prompt template — grounding, citation, refusal
- Connecting vector store to LLM in a clean pipeline
- Handling the case when no relevant context is found
- Source citation — tracking which chunks supported the answer
- Conversation memory in RAG — multi-turn Q&A over documents
- Advanced RAG: HyDE, query expansion, and parent-child retrieval
- End-to-end evaluation — faithfulness, relevance, groundedness

### Goal by End of Day

Build a complete RAG pipeline that takes a natural language question,
retrieves relevant chunks, generates a grounded answer with source
citations, and refuses to answer when the context does not contain
the information. Evaluate it systematically.

In [ ]:
## Run once
## !pip install sentence-transformers faiss-cpu openai anthropic -q

import os
import re
import json
import time
import pickle
import hashlib
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer

try:
    import faiss
    FAISS_AVAILABLE = True
except ImportError:
    FAISS_AVAILABLE = False

# ----------------------------------------------------------------
# LLM caller — same pattern from Day 12
# Falls back gracefully if no API key is set
# ----------------------------------------------------------------

def call_llm(prompt, system='You are a helpful assistant.',
             temperature=0.2, max_tokens=600):
    if os.environ.get('OPENAI_API_KEY'):
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model='gpt-3.5-turbo',
            messages=[{'role': 'system', 'content': system},
                      {'role': 'user',   'content': prompt}],
            temperature=temperature, max_tokens=max_tokens,
        )
        return resp.choices[0].message.content

    elif os.environ.get('ANTHROPIC_API_KEY'):
        import anthropic
        client = anthropic.Anthropic()
        resp = client.messages.create(
            model='claude-3-haiku-20240307', max_tokens=max_tokens,
            system=system,
            messages=[{'role': 'user', 'content': prompt}],
        )
        return resp.content[0].text

    else:
        # Mock mode — shows structure without real LLM call
        return (
            '[MOCK — set OPENAI_API_KEY or ANTHROPIC_API_KEY]\n\n'
            'Based on the provided context:\n'
            'The answer would appear here, grounded in the retrieved passages.\n'
            'Sources: [Source 1], [Source 2]'
        )

provider = (
    'openai'    if os.environ.get('OPENAI_API_KEY')    else
    'anthropic' if os.environ.get('ANTHROPIC_API_KEY') else
    'mock'
)
print(f'LLM provider : {provider}')
print('Ready.')

In [ ]:
# ----------------------------------------------------------------
# Rebuild the VectorStore and corpus from Day 14
# (Copy-pasted here so this notebook is self-contained)
# If you saved the store yesterday, you can load it instead:
#   store = VectorStore.load('/tmp/rag_store')
# ----------------------------------------------------------------

@dataclass
class Document:
    text     : str
    metadata : Dict = field(default_factory=dict)

@dataclass
class Chunk:
    text     : str
    metadata : Dict = field(default_factory=dict)
    chunk_id : str  = ''
    def __post_init__(self):
        if not self.chunk_id:
            self.chunk_id = hashlib.md5(self.text.encode()).hexdigest()[:8]

def recursive_chunker(doc, chunk_size=400, overlap=60):
    separators = ['\n\n', '\n', '. ', '! ', '? ', ' ', '']
    def split_text(text, sep_idx=0):
        if len(text) <= chunk_size or sep_idx >= len(separators):
            return [text] if text.strip() else []
        sep   = separators[sep_idx]
        parts = text.split(sep) if sep else list(text)
        merged, current = [], ''
        for part in parts:
            candidate = (current + sep + part).strip() if current else part.strip()
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current: merged.append(current)
                if len(part) > chunk_size:
                    merged.extend(split_text(part, sep_idx + 1))
                    current = ''
                else:
                    current = part.strip()
        if current: merged.append(current)
        return [m for m in merged if m.strip()]
    raw_chunks = split_text(doc.text)
    chunks = []
    for i, text in enumerate(raw_chunks):
        if i > 0 and overlap > 0:
            text = raw_chunks[i-1][-overlap:].strip() + ' ' + text
        chunks.append(Chunk(text=text.strip(),
                            metadata={**doc.metadata, 'chunk_index': i,
                                       'chunk_strategy': 'recursive'}))
    return chunks

class VectorStore:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.embed_model = SentenceTransformer(model_name)
        self.dim         = self.embed_model.get_sentence_embedding_dimension()
        self.chunks      = []
        self.id_to_idx   = {}
        if FAISS_AVAILABLE:
            self.index = faiss.IndexFlatIP(self.dim)
        else:
            self._vectors = None

    def ingest(self, documents, chunk_size=400, overlap=60):
        all_chunks = []
        for doc in documents:
            all_chunks.extend(recursive_chunker(doc, chunk_size, overlap))
        texts   = [c.text for c in all_chunks]
        vectors = self.embed_model.encode(texts, normalize_embeddings=True,
                                          show_progress_bar=True).astype(np.float32)
        start = len(self.chunks)
        for i, chunk in enumerate(all_chunks):
            self.id_to_idx[chunk.chunk_id] = start + i
            self.chunks.append(chunk)
        if FAISS_AVAILABLE:
            self.index.add(vectors)
        else:
            self._vectors = vectors if self._vectors is None else np.vstack([self._vectors, vectors])
        print(f'Indexed {len(self.chunks)} chunks.')

    def search(self, query, top_k=5, filter_metadata=None):
        q_vec = self.embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
        k_ret = top_k * 5 if filter_metadata else top_k
        if FAISS_AVAILABLE:
            scores, ids = self.index.search(q_vec, k_ret)
            scores, ids = scores[0], ids[0]
        else:
            all_s = self._vectors @ q_vec[0]
            ids   = np.argsort(all_s)[::-1][:k_ret]
            scores = all_s[ids]
        results = []
        for score, idx in zip(scores, ids):
            if idx < 0 or idx >= len(self.chunks): continue
            chunk = self.chunks[idx]
            if filter_metadata:
                if not all(chunk.metadata.get(k) == v for k, v in filter_metadata.items()): continue
            results.append({'rank': len(results)+1, 'score': float(score),
                             'text': chunk.text, 'metadata': chunk.metadata,
                             'chunk_id': chunk.chunk_id})
            if len(results) == top_k: break
        return results

    def __len__(self): return len(self.chunks)


# ----------------------------------------------------------------
# Same 6-document corpus from Day 14
# ----------------------------------------------------------------

RAW_DOCUMENTS = [
    Document(text="""Machine Learning Fundamentals

Machine learning is a branch of artificial intelligence that enables computers to learn from data without being explicitly programmed. Instead of writing rules by hand, we train models on examples.

There are three main types of machine learning. Supervised learning uses labeled data where the correct answer is known. The model learns a mapping from inputs to outputs. Common algorithms include linear regression, decision trees, and neural networks.

Unsupervised learning works with unlabeled data. The model discovers hidden structure without guidance. Clustering groups similar examples together. Dimensionality reduction compresses data while preserving important structure.

Overfitting occurs when a model learns the training data too well, including noise and outliers. It performs well on training data but fails on new examples. Regularization, dropout, and early stopping are standard techniques to prevent overfitting.

The bias-variance tradeoff describes the tension between two sources of error. High bias means the model is too simple and underfits. High variance means the model is too complex and overfits.""",
             metadata={'source': 'ml_fundamentals.txt', 'title': 'ML Fundamentals', 'domain': 'ml'}),

    Document(text="""Deep Learning and Neural Networks

Deep learning uses neural networks with many layers. Each layer learns increasingly abstract representations of the input data.

Backpropagation is the algorithm that trains neural networks. It computes the gradient of the loss with respect to every weight using the chain rule. Gradient descent then updates the weights to reduce the loss.

Convolutional Neural Networks (CNNs) are specialized for image data. Convolutional layers apply learnable filters that detect local patterns like edges and textures. Pooling layers reduce spatial dimensions.

Transfer learning reuses a model pretrained on a large dataset as the starting point for a new task. Fine-tuning the pretrained model on task-specific data is much more efficient than training from scratch.""",
             metadata={'source': 'deep_learning.txt', 'title': 'Deep Learning', 'domain': 'ml'}),

    Document(text="""The Transformer Architecture

The Transformer was introduced in the 2017 paper Attention Is All You Need. It replaced recurrent networks by relying entirely on attention mechanisms.

Self-attention is the core mechanism. For each token in a sequence, self-attention computes a weighted sum of all other tokens' values. The weights are determined by how relevant each other token is, measured by the dot product of query and key vectors.

Multi-head attention runs several attention operations in parallel. Each head learns to attend to different types of relationships. One head might track syntactic dependencies while another captures semantic similarity.

BERT uses only the encoder stack with bidirectional attention. It is pretrained on masked language modeling. GPT uses only the decoder stack with causal attention for text generation.""",
             metadata={'source': 'transformer_architecture.txt', 'title': 'Transformers', 'domain': 'nlp'}),

    Document(text="""Python Best Practices for Data Science

Virtual environments should always be used. They isolate project dependencies and prevent version conflicts. Use venv or conda to create an isolated Python installation per project.

Vectorized operations with NumPy and Pandas are dramatically faster than Python for-loops. Avoid iterating over DataFrame rows. Use .apply(), .map(), or vectorized arithmetic directly on arrays.

Memory efficiency matters for large datasets. Use generators instead of lists for large sequences. Load data in chunks with pandas chunksize parameter. Consider Dask or Polars for datasets that do not fit in memory.

Data pipelines should be reproducible. Set random seeds. Log all hyperparameters and results with tools like MLflow or Weights and Biases. Version your datasets alongside your code.""",
             metadata={'source': 'python_best_practices.txt', 'title': 'Python Best Practices', 'domain': 'programming'}),

    Document(text="""Large Language Models: A Practical Guide

Large language models are Transformer-based neural networks trained on massive text corpora using next-token prediction. Models like GPT-4, Claude, and Llama have billions of parameters.

The training process has three stages. Pretraining on internet-scale text teaches the model language structure and world knowledge. Supervised fine-tuning on human-written examples teaches instruction following. RLHF aligns the model to human preferences for helpfulness and safety.

Temperature controls the randomness of generation. Low temperature makes output deterministic and factual. High temperature increases diversity and creativity.

Fine-tuning adapts a pretrained LLM to a specific domain or task. LoRA and QLoRA allow fine-tuning large models with limited GPU memory by training only small low-rank adapter matrices.""",
             metadata={'source': 'llm_guide.txt', 'title': 'LLM Guide', 'domain': 'nlp'}),

    Document(text="""Retrieval-Augmented Generation (RAG)

RAG combines retrieval with language generation. It was introduced by Lewis et al. in 2020. RAG solves the knowledge cutoff problem and reduces hallucinations by grounding generation in retrieved documents.

In a RAG pipeline, user queries are embedded and used to retrieve semantically similar chunks from a vector store. The retrieved chunks are inserted into the LLM prompt as context. The model generates an answer grounded in the provided evidence.

Chunking strategy is critical. Chunks should be large enough to contain complete thoughts but small enough to be specific. Overlapping chunks ensure that information at chunk boundaries is not lost.

Advanced RAG techniques include HyDE which generates a hypothetical answer and uses it as the query. Re-ranking uses a cross-encoder model to re-score the initial retrieval candidates for higher precision.

Evaluation of RAG systems uses metrics like context recall, faithfulness, and answer relevance. The RAGAS framework provides automated evaluation using an LLM-as-judge approach.""",
             metadata={'source': 'rag_overview.txt', 'title': 'RAG Overview', 'domain': 'nlp'}),
]

print('Building vector store...')
store = VectorStore()
store.ingest(RAW_DOCUMENTS)
print(f'Ready. {len(store)} chunks indexed.')

---

## Part 1 — The RAG Prompt Template

The prompt template is the bridge between retrieval and generation.
It must tell the LLM:

```
1. What role to play        → you are a document Q&A assistant
2. What data to trust       → ONLY use the provided context
3. What to do when unsure   → say 'I don't know' rather than guess
4. How to cite sources      → reference [Source: filename] in answer
5. What format to use       → concise, factual, with citations

The three behaviors the system prompt must enforce:

  Groundedness  : answer must come from context, not model memory
  Citation      : every factual claim should reference its source
  Refusal       : if context does not contain the answer, say so
                  do NOT hallucinate — this is the whole point of RAG
```

### The Prompt Structure

```
[SYSTEM]
You are a precise document Q&A assistant...
Rules: only use context, cite sources, refuse if unknown

[USER]
Context:
──────────────────────
[Source: ml_fundamentals.txt]
<chunk 1 text>

[Source: deep_learning.txt]
<chunk 2 text>
──────────────────────

Question: What is overfitting?

Answer:
```

In [ ]:
# ----------------------------------------------------------------
# RAG system prompt — the grounding contract
# ----------------------------------------------------------------

RAG_SYSTEM_PROMPT = """
You are a precise document Q&A assistant. Your job is to answer
questions using ONLY the information in the provided context sections.

Rules you must follow:
1. Base your answer EXCLUSIVELY on the provided context.
   Do not use any outside knowledge or make up information.
2. After each factual claim, cite the source in parentheses:
   (Source: filename.txt)
3. If the context does not contain enough information to answer
   the question, respond with:
   'I cannot answer this question based on the available documents.'
   Do NOT guess or hallucinate.
4. Be concise and factual. Avoid unnecessary preamble.
5. If multiple sources support the same point, cite all of them.
""".strip()


def build_rag_prompt(question: str, context_chunks: List[Dict]) -> str:
    """
    Assemble the full RAG prompt from a question and retrieved chunks.
    """
    # Format context block
    context_parts = []
    for chunk in context_chunks:
        source = chunk['metadata'].get('source', 'unknown')
        context_parts.append(f"[Source: {source}]\n{chunk['text']}")

    context_block = '\n\n'.join(context_parts)

    prompt = f"""Context:
──────────────────────────────────────────
{context_block}
──────────────────────────────────────────

Question: {question}

Answer:"""
    return prompt


# Preview the prompt for one question
test_q    = 'What is overfitting and how can I prevent it?'
chunks    = store.search(test_q, top_k=2)
prompt    = build_rag_prompt(test_q, chunks)

print('RAG Prompt Preview')
print('=' * 65)
print('SYSTEM:')
print(RAG_SYSTEM_PROMPT[:200] + '...')
print()
print('USER:')
print(prompt)

---

## Part 2 — The Complete RAG Pipeline

Now we assemble the full end-to-end pipeline:

```
question
    │
    ├── embed query
    │
    ├── retrieve top-k chunks from VectorStore
    │
    ├── check relevance score threshold
    │       if max score < threshold → return 'no relevant documents found'
    │
    ├── build_rag_prompt(question, chunks)
    │
    ├── call_llm(prompt, system=RAG_SYSTEM_PROMPT)
    │
    └── return {answer, sources, chunks, scores}
```

In [ ]:
# ----------------------------------------------------------------
# The complete RAG pipeline as a clean class
# ----------------------------------------------------------------

class RAGPipeline:
    """
    End-to-end RAG pipeline.

    query → retrieve → prompt → generate → answer + sources
    """

    def __init__(
        self,
        vector_store       : VectorStore,
        top_k              : int   = 3,
        score_threshold    : float = 0.30,
        system_prompt      : str   = RAG_SYSTEM_PROMPT,
        temperature        : float = 0.1,
        max_tokens         : int   = 500,
    ):
        self.store          = vector_store
        self.top_k          = top_k
        self.score_threshold= score_threshold
        self.system_prompt  = system_prompt
        self.temperature    = temperature
        self.max_tokens     = max_tokens

    def answer(self, question: str,
               filter_metadata: Optional[Dict] = None) -> Dict:
        """
        Answer a question using RAG.

        Returns:
          answer          : str  — the generated answer
          sources         : list — unique source filenames cited
          retrieved_chunks: list — raw retrieval results
          retrieved       : bool — whether relevant chunks were found
          latency_ms      : float
        """
        t0 = time.time()

        # Step 1: Retrieve
        chunks = self.store.search(
            question,
            top_k           = self.top_k,
            filter_metadata = filter_metadata,
        )

        # Step 2: Check if retrieval found anything relevant
        if not chunks or chunks[0]['score'] < self.score_threshold:
            return {
                'answer'           : 'I cannot answer this question based on the available documents.',
                'sources'          : [],
                'retrieved_chunks' : chunks,
                'retrieved'        : False,
                'latency_ms'       : (time.time() - t0) * 1000,
            }

        # Step 3: Build prompt
        prompt = build_rag_prompt(question, chunks)

        # Step 4: Generate
        answer = call_llm(
            prompt,
            system      = self.system_prompt,
            temperature = self.temperature,
            max_tokens  = self.max_tokens,
        )

        # Step 5: Extract cited sources
        sources = list(dict.fromkeys(
            r['metadata']['source'] for r in chunks
        ))

        return {
            'answer'           : answer,
            'sources'          : sources,
            'retrieved_chunks' : chunks,
            'retrieved'        : True,
            'latency_ms'       : (time.time() - t0) * 1000,
        }

    def pretty_print(self, result: Dict):
        """Print a RAG result in a readable format."""
        print('Answer')
        print('=' * 65)
        print(result['answer'])
        print()
        print(f'Sources used ({len(result["sources"])}):')
        for src in result['sources']:
            print(f'  • {src}')
        print()
        print(f'Retrieved: {result["retrieved"]}  |  '
              f'Latency: {result["latency_ms"]:.0f}ms  |  '
              f'Chunks used: {len(result["retrieved_chunks"])}')
        print()
        if result['retrieved_chunks']:
            print('Top retrieved chunk:')
            top = result['retrieved_chunks'][0]
            print(f'  [{top["score"]:.3f}] {top["metadata"]["source"]}')
            print(f'  "{top["text"][:120]}..."')


# Build the pipeline
rag = RAGPipeline(store, top_k=3, score_threshold=0.30)
print('RAG pipeline ready.')

In [ ]:
# ----------------------------------------------------------------
# Test the RAG pipeline on several questions
# ----------------------------------------------------------------

test_questions = [
    # Questions the corpus CAN answer
    'What is overfitting and what techniques prevent it?',
    'How does self-attention work?',
    'What are the three stages of training a large language model?',
    'How do I use memory efficiently in Python with large datasets?',
    # Question the corpus CANNOT answer → should trigger refusal
    'What is the best restaurant in Tokyo?',
]

for question in test_questions:
    print(f'Question: "{question}"')
    result = rag.answer(question)
    rag.pretty_print(result)
    print('─' * 65)
    print()

---

## Part 3 — Multi-Turn RAG: Conversation Memory

A single-turn RAG answers one question at a time. But users often
ask follow-up questions that reference previous answers:

```
User : What is overfitting?
RAG  : Overfitting is when a model...
User : What techniques prevent it?   ← references previous topic
User : Can you give an example?      ← references previous answer
```

To handle this, the pipeline must maintain conversation history
and optionally rephrase follow-up questions with context before
running retrieval.

```
Two strategies for multi-turn RAG:

Strategy 1: Pass full history to LLM
  → Send all previous Q&A pairs as message history
  → LLM uses conversation context to interpret follow-ups
  → Simple but uses more tokens on every turn

Strategy 2: Query rewriting
  → Use a small LLM call to rewrite the follow-up question
    into a standalone question before retrieval
  → 'What techniques prevent it?' + history
    → 'What techniques prevent overfitting in machine learning?'
  → Better retrieval accuracy, one extra LLM call per turn
```

In [ ]:
# ----------------------------------------------------------------
# Multi-turn RAG with conversation history
# ----------------------------------------------------------------

class ConversationalRAG:
    """
    Multi-turn RAG that maintains conversation history.
    Uses query rewriting to make follow-up questions self-contained.
    """

    def __init__(self, rag_pipeline: RAGPipeline):
        self.rag     = rag_pipeline
        self.history = []   # list of (question, answer) tuples

    def _rewrite_query(self, question: str) -> str:
        """
        Rewrite a follow-up question as a standalone question
        using the conversation history for context.
        """
        if not self.history:
            return question   # first turn — no rewriting needed

        history_text = '\n'.join(
            f'Q: {q}\nA: {a[:150]}...'
            for q, a in self.history[-3:]   # last 3 turns only
        )

        rewrite_prompt = f"""
Given this conversation history:
{history_text}

Rewrite the following follow-up question as a complete, standalone question
that includes all necessary context from the history.
Return ONLY the rewritten question. No explanation.

Follow-up question: {question}
Standalone question:"""

        rewritten = call_llm(
            rewrite_prompt,
            system      = 'You rewrite questions to be self-contained.',
            temperature = 0,
            max_tokens  = 100,
        ).strip()

        return rewritten

    def chat(self, question: str) -> Dict:
        """Process one turn of the conversation."""
        # Step 1: rewrite if follow-up
        standalone_q = self._rewrite_query(question)

        # Step 2: run RAG on standalone question
        result = self.rag.answer(standalone_q)

        # Step 3: store in history
        self.history.append((question, result['answer']))

        result['original_question']  = question
        result['rewritten_question'] = standalone_q
        return result

    def reset(self):
        """Clear conversation history."""
        self.history = []


# Simulate a multi-turn conversation
conv_rag = ConversationalRAG(rag)

conversation = [
    'What is overfitting in machine learning?',
    'What techniques are used to prevent it?',
    'How is that different from the bias-variance tradeoff?',
]

print('Multi-Turn Conversational RAG')
print('=' * 65)

for turn, question in enumerate(conversation, 1):
    result = conv_rag.chat(question)

    print(f'\nTurn {turn}')
    print(f'User asked     : "{result["original_question"]}"')
    if result['rewritten_question'] != result['original_question']:
        print(f'Rewritten to   : "{result["rewritten_question"]}"')
    print(f'Answer         : {result["answer"][:200]}...')
    print(f'Sources        : {result["sources"]}')

---

## Part 4 — Advanced RAG: HyDE and Query Expansion

Standard RAG embeds the user's question and retrieves by similarity.
But questions and answers live in different semantic spaces:

```
Question : 'What prevents overfitting?'       ← interrogative style
Document : 'Regularization and dropout are   ← declarative style
            techniques that prevent...'        

These are semantically similar but the embedding distance is larger
than between two declarative sentences on the same topic.
```

### HyDE — Hypothetical Document Embeddings

```
Step 1 : Use LLM to generate a hypothetical answer to the question
         'What prevents overfitting?'
         → 'Overfitting can be prevented using regularization
             techniques such as L1/L2 penalty, dropout, and
             early stopping during training...'

Step 2 : Embed the hypothetical answer (not the original question)

Step 3 : Use that embedding for retrieval
         → now comparing answer-style text to answer-style documents
         → embedding distance is smaller → better retrieval

Tradeoff : one extra LLM call per query. Worth it for complex questions.
```

### Query Expansion

```
Generate N paraphrases of the question.
Retrieve for each paraphrase.
Merge results and deduplicate by chunk_id.
Re-rank the merged set.

Increases recall at the cost of N× retrieval calls.
Useful when the question is ambiguous or could be phrased many ways.
```

In [ ]:
# ----------------------------------------------------------------
# HyDE implementation
# ----------------------------------------------------------------

def hyde_search(question: str, store: VectorStore, top_k=3) -> List[Dict]:
    """
    Hypothetical Document Embedding retrieval.

    1. Generate a hypothetical answer to the question
    2. Embed the hypothetical answer
    3. Use that embedding for retrieval
    """
    hyde_prompt = f"""
Write a short, factual paragraph that directly answers the following question.
Write as if you are the document that contains the answer.
Be specific and use technical language. 3-4 sentences maximum.

Question: {question}
Answer paragraph:"""

    hypothetical_answer = call_llm(
        hyde_prompt,
        temperature = 0.3,
        max_tokens  = 150,
    )

    # Use the hypothetical answer as the retrieval query
    results = store.search(hypothetical_answer, top_k=top_k)

    return hypothetical_answer, results


def query_expansion_search(question: str, store: VectorStore,
                           n_expansions=3, top_k=3) -> List[Dict]:
    """
    Retrieve using multiple paraphrases of the question.
    Merges and deduplicates results by chunk_id.
    """
    expansion_prompt = f"""
Generate {n_expansions} different ways to ask the following question.
Use different vocabulary and phrasing but preserve the meaning.
Return ONLY the questions, one per line, no numbering.

Question: {question}"""

    expansions_text = call_llm(
        expansion_prompt, temperature=0.7, max_tokens=200
    )
    expansions = [q.strip() for q in expansions_text.strip().split('\n')
                  if q.strip() and not q.strip().startswith('[MOCK')]
    all_queries = [question] + expansions[:n_expansions]

    # Retrieve for each query
    seen_ids = set()
    merged   = []

    for q in all_queries:
        for r in store.search(q, top_k=top_k):
            if r['chunk_id'] not in seen_ids:
                seen_ids.add(r['chunk_id'])
                merged.append(r)

    # Re-sort by score
    merged.sort(key=lambda x: x['score'], reverse=True)
    return all_queries, merged[:top_k]


# Compare standard retrieval vs HyDE vs query expansion
question = 'What mechanisms allow neural networks to generalize to new data?'

print(f'Query: "{question}"')
print('=' * 65)

# Standard
standard = store.search(question, top_k=3)
print('\n--- Standard Retrieval ---')
for r in standard:
    print(f'  [{r["score"]:.3f}] {r["metadata"]["source"]} : {r["text"][:80]}...')

# HyDE
hyp_answer, hyde_results = hyde_search(question, store, top_k=3)
print(f'\n--- HyDE Retrieval ---')
print(f'Hypothetical answer: "{hyp_answer[:120]}..."')
print('Retrieved:')
for r in hyde_results:
    print(f'  [{r["score"]:.3f}] {r["metadata"]["source"]} : {r["text"][:80]}...')

# Query expansion
all_queries, exp_results = query_expansion_search(question, store)
print(f'\n--- Query Expansion ({len(all_queries)} queries) ---')
for q in all_queries:
    print(f'  • "{q[:70]}"')
print('Merged top results:')
for r in exp_results:
    print(f'  [{r["score"]:.3f}] {r["metadata"]["source"]} : {r["text"][:80]}...')

---

## Part 5 — End-to-End Evaluation

RAG evaluation measures three things:

```
1. Retrieval Quality
   Context Recall  : does the retrieved context contain all
                     information needed to answer the question?
   Context Precision: are the retrieved chunks relevant?
                     (no noise)

2. Generation Quality
   Faithfulness    : is every claim in the answer supported by
                     the context? (no hallucinations)
   Answer Relevance: does the answer actually address the question?

3. End-to-End
   Answer Correctness: is the answer factually correct?
                       requires ground truth answers

LLM-as-Judge approach:
  Use a second LLM call to evaluate each metric.
  Pass (question, context, answer) to the judge LLM.
  Ask it to score faithfulness on a 0-1 scale.
  This is the approach used by the RAGAS framework.
```

In [ ]:
# ----------------------------------------------------------------
# LLM-as-Judge evaluation
# ----------------------------------------------------------------

def evaluate_faithfulness(question: str, context: str,
                          answer: str) -> Dict:
    """
    Ask an LLM to judge whether the answer is faithful to the context.
    Returns a score (0-1) and reasoning.
    """
    judge_prompt = f"""
You are evaluating whether an answer is faithful to a given context.
Faithful means every factual claim in the answer is directly supported
by the context. The answer should NOT contain information from outside
the context.

Context:
{context[:800]}

Question: {question}

Answer: {answer}

Evaluate the answer's faithfulness. Return JSON with:
  "score": number from 0.0 to 1.0
    (1.0 = fully faithful, 0.0 = completely hallucinated)
  "issues": list of strings describing any unfaithful claims
  "verdict": one of "faithful", "mostly_faithful", "unfaithful"

Return ONLY the JSON object."""

    raw = call_llm(judge_prompt, temperature=0, max_tokens=200)

    # Parse JSON, handle mock responses
    try:
        raw_clean = re.sub(r'^```(?:json)?\s*', '', raw.strip())
        raw_clean = re.sub(r'\s*```$', '', raw_clean)
        return json.loads(raw_clean)
    except:
        return {'score': 0.5, 'issues': ['Could not parse judge response'],
                'verdict': 'unknown', 'raw': raw[:100]}


def evaluate_answer_relevance(question: str, answer: str) -> Dict:
    """
    Judge whether the answer actually addresses the question.
    """
    judge_prompt = f"""
Does the following answer directly address the question asked?

Question: {question}
Answer: {answer}

Return JSON with:
  "score": 0.0 to 1.0 (1.0 = directly answers the question)
  "verdict": "relevant", "partially_relevant", or "irrelevant"
  "reason": one sentence explanation

Return ONLY the JSON."""

    raw = call_llm(judge_prompt, temperature=0, max_tokens=150)
    try:
        raw_clean = re.sub(r'^```(?:json)?\s*', '', raw.strip())
        raw_clean = re.sub(r'\s*```$', '', raw_clean)
        return json.loads(raw_clean)
    except:
        return {'score': 0.5, 'verdict': 'unknown', 'reason': raw[:100]}


# Run evaluation on a test set
EVAL_QUESTIONS = [
    'What is overfitting and how can I prevent it?',
    'How does self-attention work in the Transformer?',
    'What are the three stages of training an LLM?',
    'How do I handle memory constraints in Python?',
]

print('End-to-End RAG Evaluation')
print('=' * 70)

eval_records = []

for question in EVAL_QUESTIONS:
    # Get RAG answer
    result  = rag.answer(question)
    answer  = result['answer']
    context = '\n\n'.join(r['text'] for r in result['retrieved_chunks'])

    # Evaluate
    faith = evaluate_faithfulness(question, context, answer)
    relev = evaluate_answer_relevance(question, answer)

    record = {
        'question'           : question,
        'answer'             : answer,
        'faithfulness_score' : faith.get('score', 0),
        'faithfulness_verdict': faith.get('verdict', '?'),
        'relevance_score'    : relev.get('score', 0),
        'relevance_verdict'  : relev.get('verdict', '?'),
        'sources'            : result['sources'],
    }
    eval_records.append(record)

    print(f'Q: {question[:55]}...')
    print(f'  Faithfulness : {record["faithfulness_score"]:.2f}  ({record["faithfulness_verdict"]})')
    print(f'  Relevance    : {record["relevance_score"]:.2f}  ({record["relevance_verdict"]})')
    print()

# Summary
avg_faith = np.mean([r['faithfulness_score'] for r in eval_records])
avg_relev = np.mean([r['relevance_score']    for r in eval_records])
print(f'─' * 55)
print(f'Average Faithfulness : {avg_faith:.3f}')
print(f'Average Relevance    : {avg_relev:.3f}')

In [ ]:
# ----------------------------------------------------------------
# Visualize evaluation results
# ----------------------------------------------------------------

if eval_records:
    questions_short = [r['question'][:35] + '...' for r in eval_records]
    faith_scores    = [r['faithfulness_score'] for r in eval_records]
    relev_scores    = [r['relevance_score']    for r in eval_records]

    x      = np.arange(len(questions_short))
    width  = 0.35

    fig, ax = plt.subplots(figsize=(11, 5))
    bars1 = ax.bar(x - width/2, faith_scores, width,
                   label='Faithfulness', color='steelblue', alpha=0.85)
    bars2 = ax.bar(x + width/2, relev_scores, width,
                   label='Relevance',    color='tomato',    alpha=0.85)

    ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.6, label='Target (0.8)')
    ax.set_ylim(0, 1.15)
    ax.set_xticks(x)
    ax.set_xticklabels(questions_short, rotation=15, ha='right', fontsize=8)
    ax.set_ylabel('Score (0–1)')
    ax.set_title('RAG Evaluation: Faithfulness and Answer Relevance')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

    for bar in bars1 + bars2:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.02,
                f'{h:.2f}', ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    plt.show()

In [ ]:
# ----------------------------------------------------------------
# Full demo: interactive RAG session
# Shows the complete pipeline in one clean function
# ----------------------------------------------------------------

def demo_rag(question: str):
    """
    Full RAG pipeline with verbose output.
    Shows each step so you can trace exactly what is happening.
    """
    print(f'\n{"="*65}')
    print(f'QUESTION: {question}')
    print(f'{"="*65}')

    # Step 1: Retrieve
    print('\nSTEP 1 — Retrieval')
    chunks = store.search(question, top_k=3)
    for i, c in enumerate(chunks, 1):
        print(f'  Chunk {i}: [{c["score"]:.3f}] {c["metadata"]["source"]}')
        print(f'           "{c["text"][:90]}..."')

    # Step 2: Check threshold
    print('\nSTEP 2 — Relevance check')
    top_score = chunks[0]['score'] if chunks else 0
    if top_score < 0.30:
        print(f'  Top score {top_score:.3f} < threshold 0.30 → triggering refusal')
        print('\nANSWER: I cannot answer this question based on the available documents.')
        return
    print(f'  Top score {top_score:.3f} ≥ threshold 0.30 → proceeding')

    # Step 3: Build prompt
    print('\nSTEP 3 — Prompt assembly')
    prompt = build_rag_prompt(question, chunks)
    print(f'  Prompt length: {len(prompt)} characters')

    # Step 4: Generate
    print('\nSTEP 4 — LLM generation')
    t0     = time.time()
    answer = call_llm(prompt, system=RAG_SYSTEM_PROMPT, temperature=0.1)
    ms     = (time.time() - t0) * 1000
    print(f'  Generation time: {ms:.0f}ms')

    # Step 5: Output
    print(f'\nANSWER:')
    print(answer)
    print(f'\nSOURCES: {list(dict.fromkeys(c["metadata"]["source"] for c in chunks))}')


# Run the demo
demo_rag('What is RAG and how does it help with LLM hallucinations?')
demo_rag('Who won the FIFA World Cup in 2022?')   # should trigger refusal

---

## Part 6 — RAG Design Decisions Cheatsheet

```
Decision             Options                   Recommendation
───────────────────────────────────────────────────────────────────────
Chunk size           200-800 tokens            400 tokens to start
Chunk overlap        10-20% of chunk size      60 tokens for 400-token chunks
Chunking strategy    Fixed / paragraph /       Recursive for general docs
                     recursive / semantic      Semantic for high quality
Embedding model      MiniLM / BGE / OpenAI     BGE-large or text-embedding-3
                                               MiniLM for prototyping
Vector store         FAISS / Chroma /          FAISS for local/research
                     Pinecone / Weaviate       Pinecone for production SaaS
top_k                3-10                      5 is a good starting point
Score threshold      0.25-0.40                 0.30 for balanced precision
Re-ranking           Optional cross-encoder    Add if hit_rate@1 < 0.7
Advanced retrieval   HyDE / query expansion    Add for complex questions
LLM temperature      0.0-0.3                   0.1 for factual grounded answers
Citation strategy    In-text / footnotes       In-text (Source: file.txt)
Refusal strategy     Score threshold +         Always implement — trust is key
                     system prompt instruction
Evaluation           RAGAS / LLM-as-judge      Measure faithfulness + recall
```

---

## Day 15 Summary

```
What you built today:

1.  RAG_SYSTEM_PROMPT    →  grounding contract: only context, cite, refuse
2.  build_rag_prompt()   →  assembles context + question into LLM input
3.  RAGPipeline class    →  retrieve → threshold check → generate → return
4.  ConversationalRAG    →  multi-turn with query rewriting
5.  HyDE                 →  embed hypothetical answer for better retrieval
6.  Query expansion      →  multi-query merge with deduplication
7.  LLM-as-Judge eval    →  faithfulness and relevance scoring
8.  demo_rag()           →  verbose step-by-step trace of the full pipeline

The full RAG system spans Days 13-15:
  Day 13  :  Embeddings and semantic search
  Day 14  :  Document ingestion, chunking, VectorStore
  Day 15  :  RAG prompt, generation, multi-turn, evaluation  ← TODAY

```

### Self-Check Questions

Answer these before Day 16:

1. Why should the RAG system prompt tell the LLM to refuse
   rather than guess when the context is insufficient?
2. What is query rewriting and when is it needed?
3. Your faithfulness score is 0.60. What does that mean and
   what would you change to improve it?
4. A user asks 'what did we discuss earlier about overfitting?'
   How does ConversationalRAG handle this?
5. HyDE uses one extra LLM call per query. When is this worth the cost?